Setting up data with lag and horizon

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# --- 1) Read data (all devices) ---
df_lr = spark.read.table("teams.sensorx.df_sx_800")

# --- 2) One-Hot Encode properties_deviceId ---
indexer = StringIndexer(inputCol="properties_deviceId", outputCol="deviceId_index")
df_indexed = indexer.fit(df_lr).transform(df_lr)

encoder = OneHotEncoder(inputCol="deviceId_index", outputCol="deviceId_ohe")
df_encoded = encoder.fit(df_indexed).transform(df_indexed)

# --- 2b) Create failure horizon column (time-based) ---
# For each row at time t, failure_horizon = 1 if payload_fault_state == 1
# at any point from time t to time t + H_days (looking H_days into the future).
H_days = 15  # horizon in days
H_seconds = H_days * 86400  # convert to seconds

w_horizon = (
    Window.partitionBy("properties_deviceId")
    .orderBy(F.col("timeStamp").cast("long"))   # order by epoch seconds
    .rangeBetween(0, H_seconds)                 # look ahead H_seconds
)

# count number of rows with 0 and 1 in fault_state

df_encoded = df_encoded.withColumn(
    "failure_horizon",
    F.max(F.col("payload_fault_state").cast("int")).over(w_horizon)
)

# --- 3) Create lag features (partitioned by device for correctness) ---
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "payload_xrayController_onTime",
    "delta_seconds"
]

n_lags = 20
w = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

for L in range(1, n_lags + 1):
    for col_name in sensor_cols:
        df_encoded = df_encoded.withColumn(f"{col_name}{L}", F.lag(col_name, L).over(w))

lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

# Drop rows with null lags (first n_lags rows per device)
df_clean = df_encoded.na.drop(subset=lag_cols)

# --- 4) Assemble features: sensor cols + OHE vector + lag cols ---
feature_input_cols = sensor_cols + ["deviceId_ohe"] + lag_cols
assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features_raw")
df_features = assembler.transform(df_clean) \
    .select("timeStamp", "features_raw", F.col("failure_horizon").alias("label"))

# --- 5) Chronological 80/20 train/test split ---
cutoff_epoch = df_features.selectExpr(
    "percentile_approx(CAST(timeStamp AS BIGINT), 0.8)"
).first()[0]

train = df_features.filter(F.col("timeStamp").cast("bigint") <= cutoff_epoch).cache()
test  = df_features.filter(F.col("timeStamp").cast("bigint") >  cutoff_epoch).cache()

# -- Normalization--
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,   # subtract mean (zero-mean)
    withStd=True     # divide by std  (unit variance)
 )
scaler_model = scaler.fit(train)  # learn mean & std from train only

train = scaler_model.transform(train).select("timeStamp", "features", "label").cache()
test  = scaler_model.transform(test).select("timeStamp", "features", "label").cache()

In [0]:

# --- Class weighting (give minority class higher weight) ---
label_counts = train.groupBy("label").count().collect()
total = sum(row["count"] for row in label_counts)
weight_map = {row["label"]: total / (2.0 * row["count"]) for row in label_counts}
# e.g. if 80% label=0, 20% label=1 → weights ≈ {0: 0.625, 1: 2.5}
print(f"Class weights: {weight_map}")

# Add weight column to train and test
weight_expr = F.when(F.col("label") == 1, weight_map[1]).otherwise(weight_map[0])
train_w = train.withColumn("weight", weight_expr)
test_w = test.withColumn("weight", weight_expr)


In [0]:
# Test Vala
# Raw fault_state distribution
print("Raw payload_fault_state distribution:")
df_lr.groupBy("payload_fault_state").count().orderBy("payload_fault_state").show()

# After failure horizon
print(f"Failure horizon (H={H_days} days) distribution:")
df_clean.groupBy("failure_horizon").count().orderBy("failure_horizon").show()

Logistic Regression training:

In [0]:

# --- 6) Train Logistic Regression ---
lr = LogisticRegression(maxIter=100, regParam=0.0, elasticNetParam=0.0)
LRmodel = lr.fit(train)
LRpredictions = LRmodel.transform(test)

# --- 7) Evaluate ---
auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
LRtrainingSummary = LRmodel.summary
LR_auc = auc_eval.evaluate(LRpredictions)
print(f"\nLogistic Regression with OHE + {n_lags} lags")
print(f"AUC: {LR_auc:.4f}")

# Confusion matrix
LRpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

Random forest with lag and horizon:

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# --- Class weighting (give minority class higher weight) ---
label_counts = train.groupBy("label").count().collect()
total = sum(row["count"] for row in label_counts)
weight_map = {row["label"]: total / (2.0 * row["count"]) for row in label_counts}
# e.g. if 80% label=0, 20% label=1 → weights ≈ {0: 0.625, 1: 2.5}
print(f"Class weights: {weight_map}")

# Add weight column to train and test
weight_expr = F.when(F.col("label") == 1, weight_map[1]).otherwise(weight_map[0])
train_w = train.withColumn("weight", weight_expr)
test_w = test.withColumn("weight", weight_expr)

# --- RandomForest with class weighting ---
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    numTrees=100,
    seed=42
)

auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="recallByLabel"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="fMeasureByLabel"
)

RFmodel = rf.fit(train_w)
RFpredictions = RFmodel.transform(test_w)

RFtrainingSummary = RFmodel.summary

RF_auc = auc_eval.evaluate(RFpredictions)

# Recall (per class)
RF_recall_0 = recall_eval.evaluate(RFpredictions, {recall_eval.metricLabel: 0.0})
RF_recall_1 = recall_eval.evaluate(RFpredictions, {recall_eval.metricLabel: 1.0})

# F1 score (per class)
RF_f1_0 = f1_eval.evaluate(RFpredictions, {f1_eval.metricLabel: 0.0})
RF_f1_1 = f1_eval.evaluate(RFpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nRandom Forest with {n_lags} lags (class-weighted)")
print(f"AUC: {RF_auc:.4f}")
print(f"Recall (label=0): {RF_recall_0:.4f}")
print(f"Recall (label=1): {RF_recall_1:.4f}")
print(f"F1     (label=0): {RF_f1_0:.4f}")
print(f"F1     (label=1): {RF_f1_1:.4f}")

# Confusion matrix
RFpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

In [0]:
# --- Adjust classification threshold ---
# Default threshold is 0.5. Lowering it makes the model predict more 1s,
# increasing recall on label=1 at the cost of more false positives.

from pyspark.ml.functions import vector_to_array
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Extract probability of class 1 from the probability vector
RF_with_prob = RFpredictions.withColumn(
    "prob_1", vector_to_array("probability")[1]
)

recall_t = MulticlassClassificationEvaluator(labelCol="label", predictionCol="custom_prediction", metricName="recallByLabel")
precision_t = MulticlassClassificationEvaluator(labelCol="label", predictionCol="custom_prediction", metricName="precisionByLabel")
f1_t = MulticlassClassificationEvaluator(labelCol="label", predictionCol="custom_prediction", metricName="fMeasureByLabel")

for threshold in [ 0.5, 0.45, 0.4]:
    df_t = RF_with_prob.withColumn(
        "custom_prediction",
        F.when(F.col("prob_1") >= threshold, 1.0).otherwise(0.0)
    )
    r  = recall_t.evaluate(df_t, {recall_t.metricLabel: 1.0})
    p  = precision_t.evaluate(df_t, {precision_t.metricLabel: 1.0})
    f1 = f1_t.evaluate(df_t, {f1_t.metricLabel: 1.0})
    print(f"Threshold={threshold:.2f} | Recall={r:.4f} | Precision={p:.4f} | F1={f1:.4f}")
    df_t.groupBy("label").pivot("custom_prediction").count().orderBy("label").show()

Gradient Boosted trees with lag and horizon:


In [0]:
from pyspark.ml.classification import GBTClassifier

# Use un-normalized features for GBT (tree models don't benefit from scaling)
train_gbt = df_features.filter(F.col("timeStamp").cast("bigint") <= cutoff_epoch) \
    .select("timeStamp", F.col("features_raw").alias("features"), "label")
train_gbt = train_gbt.withColumn("weight", F.when(F.col("label") == 1, weight_map[1]).otherwise(weight_map[0])).cache()

test_gbt = df_features.filter(F.col("timeStamp").cast("bigint") > cutoff_epoch) \
    .select("timeStamp", F.col("features_raw").alias("features"), "label")
test_gbt = test_gbt.withColumn("weight", F.when(F.col("label") == 1, weight_map[1]).otherwise(weight_map[0])).cache()

# Train GBT directly on raw (un-normalized) features
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    maxIter=30,
    seed=42
)

GBTmodel = gbt.fit(train_gbt)
gbtpredictions = GBTmodel.transform(test_gbt)

# --- Evaluate (same format as LR / RF) ---
GBT_auc = auc_eval.evaluate(gbtpredictions)

# Recall (per class)
GBT_recall_0 = recall_eval.evaluate(gbtpredictions, {recall_eval.metricLabel: 0.0})
GBT_recall_1 = recall_eval.evaluate(gbtpredictions, {recall_eval.metricLabel: 1.0})

# F1 score (per class)
GBT_f1_0 = f1_eval.evaluate(gbtpredictions, {f1_eval.metricLabel: 0.0})
GBT_f1_1 = f1_eval.evaluate(gbtpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nGBT with {n_lags} lags (un-normalized, class-weighted)")
print(f"AUC: {GBT_auc:.4f}")
print(f"Recall (label=0): {GBT_recall_0:.4f}")
print(f"Recall (label=1): {GBT_recall_1:.4f}")
print(f"F1     (label=0): {GBT_f1_0:.4f}")
print(f"F1     (label=1): {GBT_f1_1:.4f}")

# Confusion matrix
gbtpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

In [0]:
import mlflow

with mlflow.start_run(run_name="GBT_fault_prediction") as run:
    # Log parameters
    mlflow.log_param("model_type", "GBTClassifier")
    mlflow.log_param("maxIter", 30)
    mlflow.log_param("n_lags", n_lags)
    mlflow.log_param("H_days", H_days)

    # Log metrics
    mlflow.log_metric("AUC", GBT_auc)
    mlflow.log_metric("Recall_label_0", GBT_recall_0)
    mlflow.log_metric("Recall_label_1", GBT_recall_1)
    mlflow.log_metric("F1_label_0", GBT_f1_0)
    mlflow.log_metric("F1_label_1", GBT_f1_1)

    print(f"Experiment logged! Run ID: {run.info.run_id}")
    print(f"AUC: {GBT_auc:.4f}")
    print(f"Recall (label=1): {GBT_recall_1:.4f}")
    print(f"F1 (label=1): {GBT_f1_1:.4f}")

In [0]:
from pyspark.ml.functions import vector_to_array

# Run predictions on full test set
all_predictions = GBTmodel.transform(test_gbt)

# Join back with original data to get deviceId and actual fault_state
original_cols = df_encoded.select(
    "timeStamp", "properties_deviceId", 
    F.col("payload_fault_state").cast("int").alias("actual_fault_state")
)

timeline = (
    all_predictions
    .join(original_cols, on="timeStamp", how="inner")
    .select(
        "timeStamp",
        "properties_deviceId",
        F.col("actual_fault_state").alias("fault_now"),
        F.col("label").alias("fault_within_15d"),
        F.col("prediction").cast("int").alias("predicted_fault_within_15d"),
        vector_to_array("probability")[1].alias("prob_fault")
    )
    .orderBy("properties_deviceId", "timeStamp")
)

# Pick a device that actually has a fault transition (fault_now goes 0 → 1)
device_with_fault = (
    timeline
    .filter(F.col("fault_now") == 1)
    .select("properties_deviceId")
    .first()
)

if device_with_fault:
    device_id = device_with_fault["properties_deviceId"]
    print(f"Showing timeline for device: {device_id}")
    print(f"fault_now = actual payload_fault_state at this moment")
    print(f"fault_within_15d = label (1 if a fault occurs within 15 days)")
    print(f"predicted_fault_within_15d = model prediction\n")
    
    device_timeline = timeline.filter(F.col("properties_deviceId") == device_id)
    display(device_timeline)
else:
    print("No device with fault transition found in test set")
    display(timeline.limit(100))

Neural Network testing

In [0]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

# Input layer = 161 features, two hidden layers, output = 2 classes (binary)
layers = [161,32,2]

trainer = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=30,
    layers=layers,
    blockSize=128,
    seed=42
)

auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="recallByLabel"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="fMeasureByLabel"
)
# Use normalized data (MLP benefits from normalization, but no weightCol support)
MLPmodel = trainer.fit(train)
MLPpredictions = MLPmodel.transform(test)

# --- Evaluate (same format as RF / GBT) ---
MLP_auc = auc_eval.evaluate(MLPpredictions)

MLP_recall_0 = recall_eval.evaluate(MLPpredictions, {recall_eval.metricLabel: 0.0})
MLP_recall_1 = recall_eval.evaluate(MLPpredictions, {recall_eval.metricLabel: 1.0})

MLP_f1_0 = f1_eval.evaluate(MLPpredictions, {f1_eval.metricLabel: 0.0})
MLP_f1_1 = f1_eval.evaluate(MLPpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nMLP Neural Network with {n_lags} lags (normalized, no class weighting)")
print(f"Layers: {layers}")
print(f"AUC: {MLP_auc:.4f}")
print(f"Recall (label=0): {MLP_recall_0:.4f}")
print(f"Recall (label=1): {MLP_recall_1:.4f}")
print(f"F1     (label=0): {MLP_f1_0:.4f}")
print(f"F1     (label=1): {MLP_f1_1:.4f}")

# Confusion matrix
MLPpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()